In [ ]:
import pandas as pd


In [3]:
df = pd.read_csv('deepfake_detection_metadata_dataset.csv')
df.head()


,media_id,media_type,content_category,face_count,audio_present,lip_sync_score,visual_artifacts_score,compression_level,lighting_inconsistency_score,source_platform,generation_method,label
0,1,Image,News,2,Yes,0.76,0.24,0.06,0.23,Facebook,NaN,Real
1,2,Video,News,3,Yes,0.01,0.82,0.62,0.98,Instagram,Diffusion,Fake
2,3,Video,Political Speech,3,No,0.20,0.66,0.23,0.77,News Website,GAN,Fake
3,4,Image,Social Media,4,Yes,0.81,0.19,0.68,0.29,YouTube,NaN,Real
4,5,Video,Interview,1,Yes,0.98,0.00,0.23,0.17,Twitter,NaN,Real


In [4]:
# To see how many rows and columns the dataset has:
df.shape


(1000, 12)

In [5]:
# To see the data types of all the columns:
df.dtypes


media_id                          int64
media_type                          str
content_category                    str
face_count                        int64
audio_present                       str
lip_sync_score                  float64
visual_artifacts_score          float64
compression_level               float64
lighting_inconsistency_score    float64
source_platform                     str
generation_method                   str
label                               str
dtype: object

In [6]:
# Drop columns that won't help us predict
df = df.drop(columns=['media_id', 'generation_method'])

print("Columns remaining:", df.columns.tolist())


Columns remaining: ['media_type', 'content_category', 'face_count', 'audio_present', 'lip_sync_score', 'visual_artifacts_score', 'compression_level', 'lighting_inconsistency_score', 'source_platform', 'label']


In [7]:
# Convert 'Real' to 0 and 'Fake' to 1
df['label'] = df['label'].map({'Real': 0, 'Fake': 1})

# Check that it worked! Let's look at the first 5 rows again:
df.head()


,media_type,content_category,face_count,audio_present,lip_sync_score,visual_artifacts_score,compression_level,lighting_inconsistency_score,source_platform,label
0,Image,News,2,Yes,0.76,0.24,0.06,0.23,Facebook,0
1,Video,News,3,Yes,0.01,0.82,0.62,0.98,Instagram,1
2,Video,Political Speech,3,No,0.20,0.66,0.23,0.77,News Website,1
3,Image,Social Media,4,Yes,0.81,0.19,0.68,0.29,YouTube,0
4,Video,Interview,1,Yes,0.98,0.00,0.23,0.17,Twitter,0


In [8]:
# Convert all remaining text (categorical) columns into One-Hot Encoded numerical columns
df = pd.get_dummies(df, drop_first=True)

# Let's see how our dataset looks now!
df.head()


,face_count,lip_sync_score,visual_artifacts_score,compression_level,lighting_inconsistency_score,label,media_type_Image,media_type_Video,content_category_Interview,content_category_News,content_category_Political Speech,content_category_Social Media,audio_present_Yes,source_platform_Instagram,source_platform_News Website,source_platform_TikTok,source_platform_Twitter,source_platform_YouTube
0,2,0.76,0.24,0.06,0.23,0,True,False,False,True,False,False,True,False,False,False,False,False
1,3,0.01,0.82,0.62,0.98,1,False,True,False,True,False,False,True,True,False,False,False,False
2,3,0.20,0.66,0.23,0.77,1,False,True,False,False,True,False,False,False,True,False,False,False
3,4,0.81,0.19,0.68,0.29,0,True,False,False,False,False,True,True,False,False,False,False,True
4,1,0.98,0.00,0.23,0.17,0,False,True,True,False,False,False,True,False,False,False,True,False


In [10]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from sklearn.preprocessing import StandardScaler

# Make a list of just our numerical columns (we exclude 'label' and our new one-hot columns)
numerical_cols = ['face_count', 'lip_sync_score', 'visual_artifacts_score', 
                 'compression_level', 'lighting_inconsistency_score']

# Create the scaler
scaler = StandardScaler()

# "Fit" the scaler to our data and transform it, then replace the old columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Look at the data one more time:
df.head()


,face_count,lip_sync_score,visual_artifacts_score,compression_level,lighting_inconsistency_score,label,media_type_Image,media_type_Video,content_category_Interview,content_category_News,content_category_Political Speech,content_category_Social Media,audio_present_Yes,source_platform_Instagram,source_platform_News Website,source_platform_TikTok,source_platform_Twitter,source_platform_YouTube
0,-0.026893,0.547107,-0.576432,-1.602127,-0.635710,0,True,False,False,True,False,False,True,False,False,False,False,False
1,0.680823,-2.022276,1.332175,0.333895,1.843786,1,False,True,False,True,False,False,True,True,False,False,False,False
2,0.680823,-1.371366,0.805663,-1.014406,1.149527,1,False,True,False,False,True,False,False,False,True,False,False,False
3,1.388540,0.718400,-0.740968,0.541325,-0.437350,0,True,False,False,False,False,True,True,False,False,False,False,True
4,-0.734610,1.300793,-1.366201,-1.014406,-0.834069,0,False,True,True,False,False,False,True,False,False,False,True,False


In [12]:
from sklearn.model_selection import train_test_split

# First, separate our Target (y) from our Features (X)
y = df['label']
X = df.drop(columns=['label'])

# Now, magically split it! test_size=0.2 means 20% is held back for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"We have {X_train.shape[0]} rows for training and {X_test.shape[0]} rows for testing!")


We have 800 rows for training and 200 rows for testing!


In [13]:
from sklearn.ensemble import RandomForestClassifier

# 1. Create the model (we'll use 100 trees)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the model using our Training Data!
rf_model.fit(X_train, y_train)

print("Training Complete! The model has learned from the data.")


Training Complete! The model has learned from the data.


In [14]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Ask the model to make predictions on the test data
y_pred = rf_model.predict(X_test)

# 2. Calculate the overall Accuracy (percentage it got exactly right)
acc = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {acc * 100:.2f}%\n")

# 3. Print a detailed report card
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))


Overall Accuracy: 100.00%

Detailed Classification Report:
              precision    recall  f1-score   support

    Real (0)       1.00      1.00      1.00       113
    Fake (1)       1.00      1.00      1.00        87

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [15]:
# 1. Start fresh with the cleaned data BUT drop the cheat scores
cols_to_drop = ['label', 'lip_sync_score', 'visual_artifacts_score', 
                'compression_level', 'lighting_inconsistency_score']

X_hard = df.drop(columns=cols_to_drop)
y_hard = df['label']

# 2. Split it
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_hard, y_hard, test_size=0.2, random_state=42)

# 3. Train a new model
rf_model_hard = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_hard.fit(X_train_h, y_train_h)

# 4. Predict and evaluate!
y_pred_h = rf_model_hard.predict(X_test_h)
hard_acc = accuracy_score(y_test_h, y_pred_h)

print(f"Realistic Accuracy: {hard_acc * 100:.2f}%\n")


Realistic Accuracy: 50.50%

